# 🌳 Ternary-Bonsai-2-27B (Q2_0) on Colab — llama.cpp + Public API

Run the cells top to bottom. At the end you get a **public HTTPS URL** for the API.

- Model: `prism-ml/Ternary-Bonsai-2-27B-gguf` **Q2_0** (~7.6 GB, ternary 27B dense, hybrid attention ~75% linear)
- Runner: **llama.cpp `llama-server`** (OpenAI-compatible `/v1`) — **prebuilt Linux CUDA binary**, no compile
  (from the PrismML fork release `prism-b12128`, based on stock llama.cpp b12128 ≥ b11556,
  which is the minimum version that loads ternary Q2_0 correctly)
- Runtime needed: **T4 GPU** — the model fits **100% in VRAM**, no CPU offload
- Served model id: `bonsai-2-27b` (that's what OpenCode/the .bat uses)
- Context served: **32,768** (native 262k; raise `-c` if you want more — KV cache is tiny thanks to linear attention)

> ⚠️ This is a **reasoning model** — it thinks by default (`xhigh` effort). Give requests enough `max_tokens` or answers get cut mid-thought.
> ⚠️ Keep this browser tab open — free Colab disconnects on idle.
> ⚠️ The tunnel URL changes every time you re-run the tunnel cell (no account needed).

In [ ]:
# 1) Verify the GPU and disk
import shutil
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
g = shutil.disk_usage('/')
print(f"disk: {g.total/1e9:.0f} GB total, {g.free/1e9:.0f} GB free (need ~15 GB)")

In [ ]:
# 2) Get llama-server — PREBUILT Linux CUDA binary (no compile, ~1-2 min)
#    PrismML fork release prism-b12128-d7b2896: based on stock llama.cpp b12128
#    (>= b11556, required for ternary Q2_0) and ships Linux CUDA prebuilts for
#    several driver generations. We pick the variant matching this VM's driver.
import subprocess, re, glob, os
out = subprocess.run('nvidia-smi', shell=True, capture_output=True, text=True).stdout
m = re.search(r'Driver Version:\s*(\d+)\.', out)
major = int(m.group(1)) if m else 0
if major >= 590:
    asset = 'llama-prism-b12128-d7b2896-bin-linux-cuda-13.3-x64.tar.gz'
elif major >= 570:
    asset = 'llama-prism-b12128-d7b2896-bin-linux-cuda-12.8-x64.tar.gz'
else:
    asset = 'llama-prism-b12128-d7b2896-bin-linux-cuda-12.4-x64.tar.gz'
print(f'NVIDIA driver {major}.x -> using {asset}')
base = 'https://github.com/PrismML-Eng/llama.cpp/releases/download/prism-b12128-d7b2896/'
subprocess.run(f'curl -sL -o /content/{asset} {base}{asset}', shell=True, check=True)
subprocess.run(f'mkdir -p /content/pkg && tar -xzf /content/{asset} -C /content/pkg', shell=True, check=True)
found = glob.glob('/content/pkg/**/llama-server', recursive=True)
assert found, 'llama-server binary not found in the archive'
print('llama-server at:', found[0])
open('/content/llama_bin.txt', 'w').write(found[0])
libs = glob.glob('/content/pkg/**/libggml*.so*', recursive=True)
envline = ''
if libs:
    envline = f'LD_LIBRARY_PATH={os.path.dirname(libs[0])}'
    open('/content/llama_env.txt', 'w').write(envline)
    print('ggml libs at:', os.path.dirname(libs[0]))
subprocess.run(f'{envline} "{found[0]}" --version', shell=True)

In [ ]:
# 3) Download the ternary model — Q2_0 pack, 7.63 GB, ~2-5 min on Colab bandwidth.
#    Interrupted? Re-run with -c (continue) and it resumes.
!wget -c -q --show-progress -O /content/Ternary-Bonsai-2-27B-Q2_0.gguf https://huggingface.co/prism-ml/Ternary-Bonsai-2-27B-gguf/resolve/main/Ternary-Bonsai-2-27B-Q2_0.gguf
!ls -lh /content/*.gguf

In [ ]:
# 4) Start llama-server: whole model on GPU (-ngl 99), 32k context, port 11434
import subprocess, time, urllib.request
!mkdir -p /content/logs
LLAMA = open('/content/llama_bin.txt').read().strip()
try:
    envline = open('/content/llama_env.txt').read().strip() + ' '
except Exception:
    envline = ''
subprocess.run('pkill -f llama-server; true', shell=True)
subprocess.Popen(['bash', '-c',
    f'{envline}nohup "{LLAMA}" '
    '-m /content/Ternary-Bonsai-2-27B-Q2_0.gguf '
    '--alias bonsai-2-27b -ngl 99 -fa on -c 32768 --port 11434 '
    '> /content/logs/llama-server.log 2>&1 &'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):
    time.sleep(2)
    try:
        v = urllib.request.urlopen('http://127.0.0.1:11434/health', timeout=5).read()
        print('llama-server alive:', v.decode()[:120])
        break
    except Exception:
        pass
else:
    print(open('/content/logs/llama-server.log', errors='ignore').read()[-1200:])
    raise RuntimeError('llama-server did not start')

In [ ]:
# 5) Load the model onto the GPU and smoke-test it (~1-2 min first call)
import json, time, urllib.request
t0 = time.time()
req = urllib.request.Request(
    'http://127.0.0.1:11434/v1/chat/completions',
    data=json.dumps({'model': 'bonsai-2-27b',
                     'messages': [{'role': 'user', 'content': 'Reply with exactly: MODEL LOADED'}],
                     'max_tokens': 64}).encode(),
    headers={'Content-Type': 'application/json'})
out = json.loads(urllib.request.urlopen(req, timeout=1200).read())
print(f'load + reply took {time.time()-t0:.1f}s')
print(out['choices'][0]['message']['content'][:400])

In [ ]:
# 6) Expose the API publicly (no account needed — Cloudflare quick tunnel)
import subprocess, time, re
subprocess.run('pkill -f cloudflared; true', shell=True)
!curl -sL -o /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared
subprocess.Popen(['bash', '-c',
    '/content/cloudflared tunnel --url http://localhost:11434 '
    '--http-host-header="localhost:11434" --no-autoupdate '
    '> /content/logs/tunnel.log 2>&1'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r'https://[a-z0-9-]+\\.trycloudflare\\.com',
                  open('/content/logs/tunnel.log', errors='ignore').read())
    if m:
        url = m.group(0)
        break
if url:
    print('PUBLIC API URL:', url)
    print('OpenAI endpoint:', url + '/v1')
    print('Copy this URL — it goes into the setup .bat on your PC')
else:
    print(open('/content/logs/tunnel.log', errors='ignore').read()[-800:])
    raise RuntimeError('tunnel did not come up')

In [ ]:
# 7) Verify the public URL works end-to-end (calls back through the internet)
import json, urllib.request
PUBLIC_URL = url  # from the previous cell
req = urllib.request.Request(
    PUBLIC_URL + '/v1/chat/completions',
    data=json.dumps({'model': 'bonsai-2-27b',
                     'messages': [{'role': 'user', 'content': 'Reply with exactly: API IS LIVE'}],
                     'max_tokens': 64}).encode(),
    headers={'Content-Type': 'application/json', 'Authorization': 'Bearer ollama'})
out = json.loads(urllib.request.urlopen(req, timeout=600).read())
print(out['choices'][0]['message']['content'][:400])

## 🔌 Connect OpenCode (on your PC)

Double-click **`setup-opencode_Ornith-1.5-35B-A3B_Q4_K_M_ollama.bat`** in the repo folder:

1. Paste the tunnel URL from cell 6
2. When it asks **which model is running**, choose **`2. Ternary-Bonsai-2-27B Q2_0 (llama.cpp)`**

It writes the provider into `C:\\Users\\DELL\\.config\\opencode\\opencode.json` (merges, never
overwrites). In OpenCode pick the provider **“Bonsai 2 27B (Colab)”** → model `bonsai-2-27b`.

Other OpenAI-compatible clients: `base_url = https://<tunnel>/v1`, `api_key = ollama`
(any non-empty string — llama-server ignores auth), model id `bonsai-2-27b`.

### Notes & limits
- **Whole model on the T4** (7.6 GB of ~14 GB usable) — no CPU offload, unlike the Ornith setup.
- **Thinking model**: default reasoning effort is `xhigh` — budget `max_tokens` generously
  (a 256-token cap ends generation mid-thought, before any answer). Use the recommended
  sampling for thinking mode: `temp 1.0, top_p 0.95, top_k 20, min_p 0.05`.
- Context served: **32k** (native 262k) — same guidance as the Ornith notebook: keep OpenCode
  sessions under ~30k tokens.
- The tunnel + server die when the Colab session stops — re-run cells 2→7 to bring it back
  (the binary + model are gone with the VM — binary ~1–2 min, model ~2–5 min).
- New session → new tunnel URL → just re-run the .bat with the new URL.
- No authentication: treat the tunnel URL as a secret.